# ABテスト入門（５）（吉村）

R で演習するための Google Colab notebook です。
Googleアカウントにログインした状態でブラウザから利用できます。

**コードは自由に編集して構いません。**

コメント加筆や修正しながらの利用を奨めます。同時に閲覧する他の利用者とも編集内容は共有されません。
誤ってコード等を消した場合、再び本ページにアクセスし直せばデフォルトから再開できます。
Google driveに本ページ（ipynbファイル）を保存すると編集内容を引き継ぐことができます。

ランタイムは R、ハードウェアは CPU を前提とします。
問い合わせ先：yoshimura at cc.kyoto-su.ac.jp

# コード演習で学ぶこと


*   分散分析
*   可視化方法
*   演習（課題）


# メインコード

In [ ]:
# K群のランダム割り当て
set.seed(1)                                                  # シード
n <- 300
K <- 3
group <- sample(rep(1:K, each = n / K))                       # 各群に同人数をランダム割り当て
print(group)

In [ ]:
# 群ごとに平均値が異なるとします
# 今回は分析分析の説明用のデータ生成とします（便宜上、潜在結果は明示しない形でデータ生成）
set.seed(1)
mu <- c(0.00, 0.30, 0.60)                                     # 群ごとの期待値を設定
y <- rnorm(n, mean = mu[group], sd = 1)                       # データを生成（群ごとに平均が違う）

by_grp <- split(y, group)                                     # split() により、データを群ごとのベクトルに分割
print(by_grp[3])                                              # 例えば群3を見てみる

In [ ]:
# sapply() により群ごとに関数処理
n_g   <- sapply(by_grp, length)                               # 各群の人数
mean_g<- sapply(by_grp, mean)                                 # 各群の平均
sd_g  <- sapply(by_grp, sd)                                   # 各群の標準偏差
print(n_g)
print(mean_g)
print(sd_g)

# data frameの形式で結果をまとめて保存
summary_table <- data.frame(group = 1:K, n = n_g, mean = mean_g, sd = sd_g)
print(summary_table)

# 群ごとの平均値や標準偏差の違いが分かります（当然、期待値が反映されている）

In [ ]:
# 3群の比較：箱ひげ図による可視化
# 群ごとに箱ひげ図を出す（boxplot(data ~ factor(group index))）
boxplot(y ~ factor(group), names = c("A", "B", "C"), ylab = "value", main = "Boxplots of 3 groups")
points(1:3, tapply(y, group, mean), pch = 19)

In [ ]:
# 3群のバイオリンプロット（※参考資料）
set.seed(1)
n <- 300
g <- rep(1:3, each = n / 3)
y <- rnorm(n, mean = c(0, 0.2, 0.5)[g], sd = 1)

plot(NA, xlim = c(0.5, 3.5), ylim = range(y), xaxt = "n", xlab = "group", ylab = "value", main = "Violin-like (base)")
axis(1, at = 1:3, labels = c("A", "B", "C"))
for (i in 1:3) { d <- density(y[g == i]); xL <- i - d$y / max(d$y) * 0.35; xR <- i + d$y / max(d$y) * 0.35; polygon(c(xL, rev(xR)), c(d$x, rev(d$x)), col = rgb(0, 0, 1, 0.25), border = NA) }
points(1:3, tapply(y, g, mean), pch = 19)

In [ ]:
# F分布の可視化：df1 = df2 = nu の重ね描き
x <- seq(0, 5, length = 400)

plot(x, df(x, df1 = 1, df2 = 1), type = "l", ylab = "density", main = "F(df1 = df2 = ν)", ylim = c(0, 1.2))
lines(x, df(x, df1 = 3, df2 = 3), col = "red")
lines(x, df(x, df1 = 10, df2 = 10), col = "darkgreen")
lines(x, df(x, df1 = 20, df2 = 20), col = "purple")
legend("topright", legend = c("ν=1", "ν=3", "ν=10", "ν=20"), col = c("black", "red", "darkgreen", "purple"), lty = 1, bty = "n")

In [ ]:
# F分布の可視化：df1 = 2で固定するケース
x <- seq(0, 5, length = 400)
plot(x, df(x, df1 = 2, df2 = 10), type = "l", ylab = "density", main = "F(df1=2, df2=10/30/120)")
lines(x, df(x, df1 = 2, df2 = 30), col = "red")
lines(x, df(x, df1 = 2, df2 = 120), col = "darkgreen")
legend("topright", legend = c("df2=10", "df2=30", "df2=120"), col = c("black", "red", "darkgreen"), lty = 1, bty = "n")

In [ ]:
# 今回は aov による分散分析を紹介します（標準ライブラリ statsより）
set.seed(1)

n <- 300
g <- rep(1:3, each = n / 3)
y <- rnorm(n, mean = c(0, 0.3, 0.6)[g], sd = 1)

# 分散分析の結果を総出力する関数 "aov" を活用します
fit <- aov(y ~ factor(g))   # "gを群のindexと見なして、データyを分散分析せよ"と読む
print(summary(fit))         # summary() で簡潔にして出力する

In [ ]:
# F値とそのp値を見てみる
# p値：仮説の下でデータ（あるいはデータよりも極端な値）が得られる確率

s <- summary(fit)[[1]]
cat("F = ", s[1, "F value"], "  p = ", s[1, "Pr(>F)"], "\n")

# p値が0.05より小さければ、帰無仮説を棄却。このとき、「群間の期待値には違いがある」と判断
# うまく言い当てられているでしょうか？確認してください

# 演習（課題１）

In [ ]:
# 動画広告の滞在時間(単位：秒)のABテストを考えます
# 以下は、動画3つ（A, B, C）として3群ある例で、上記コードを前提に作成されたサンプルコードです
# それぞれの動画の滞在時間の期待値は21秒、22秒、23秒と設定されています
# これらの仮定の下で架空のデータが生成され、箱ひげ図、F値・p値を出力するコードになっています

# 問題：コードを次のように修正して、F値とp値を出力してください
# テストする動画は4つ（A, B, C, D）として、4群あるとしてください
# それぞれの動画の滞在時間の期待値を25秒, 26秒, 24秒, 25.05秒とします
# なお、シードは1のまま固めておいてください
# エラーがないように作成してください

# さて、4群の期待値は等しいという仮説は棄却されるでしょうか？

set.seed(1)                                                   # シード
n <- 300
K <- 3
group <- sample(rep(1:K, each = n / K))                       # 各群に同人数をランダム割り当て
mu <- c(21, 22, 23)                                           # 群ごとの期待値を設定
y <- rnorm(n, mean = mu[group], sd = 1)
by_grp <- split(y, group)
n_g   <- sapply(by_grp, length)
mean_g<- sapply(by_grp, mean)
sd_g  <- sapply(by_grp, sd)
summary_table <- data.frame(group = 1:K, n = n_g, mean = mean_g, sd = sd_g)
boxplot(y ~ factor(group), names = c("A", "B", "C"), ylab = "value", main = "Boxplots")
fit <- aov(y ~ factor(group))
s <- summary(fit)[[1]]
cat("F = ", s[1, "F value"], "  p = ", s[1, "Pr(>F)"], "\n")